# TravelMate AI — Manual Training Runbook

Notebook tổng hợp toàn bộ quy trình train theo từng ô Run của Google Colab: thiết lập môi trường, kiểm tra GPU, chuẩn bị split v12, validate, dry-run, train QLoRA, sinh prediction, đánh giá và regression.

> **An toàn:** ô train thật mặc định bị khóa. Không đổi adapter production trực tiếp; luôn ghi ra một thư mục candidate mới. `test` và `challenge` chỉ dùng đánh giá, không đưa vào train.

## Ô 1 — Chuẩn bị repository và dependency

Chọn GPU tại **Runtime → Change runtime type → GPU**. Cell hỗ trợ clone Git hoặc giải nén bundle `/content/travelmate-ai-colab.zip`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/trongnd16092005/travelmate-ai.git"  # @param {type:"string"}
BRANCH = "feature/ai-itinerary-generation"  # @param {type:"string"}
REPO_DIR = Path("/content/travelmate-ai")
BUNDLE_PATH = Path("/content/travelmate-ai-colab.zip")

if BUNDLE_PATH.exists() and not REPO_DIR.exists():
    subprocess.run(["unzip", "-q", "-o", str(BUNDLE_PATH), "-d", "/content"], check=True)
elif not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

SERVICE_DIR = REPO_DIR / "services" / "ai-service"
os.chdir(SERVICE_DIR)
if sys.version_info < (3, 12):
    raise RuntimeError(f"TravelMate yêu cầu Python >= 3.12; runtime hiện tại là {sys.version.split()[0]}")
subprocess.run(["git", "lfs", "install"], check=True)
subprocess.run(["git", "lfs", "pull"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[training,dev]"], check=True)
print("Working directory:", Path.cwd())

## Ô 2 — Kiểm tra GPU

Dừng tại đây nếu Colab chưa nhận GPU NVIDIA.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Hãy chọn Runtime > Change runtime type > GPU rồi chạy lại.")
properties = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(properties.total_memory / 1024**3, 2))
subprocess.run(["nvidia-smi"], check=False)

## Ô 3 — Cấu hình lần train

Mặc định tái lập đúng lượt r3: bắt đầu từ adapter r2, dùng 684 mẫu grounded và learning rate `5e-6`. Chế độ `new_candidate` bắt đầu từ production r3 và phải dùng run name mới. Có thể lưu toàn bộ output vào Google Drive để không mất khi Colab reset.

In [ ]:
RUN_MODE = "reproduce_v12_r3"  # @param ["reproduce_v12_r3", "new_candidate"]
RUN_NAME = "v12-r3-reproduction-01"  # @param {type:"string"}
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/TravelMateTraining"  # @param {type:"string"}
R2_ADAPTER_PATH = "artifacts/travelmate-qwen3-4b-lora-v12-grounded-conversation-r2"  # @param {type:"string"}
REGRESSION_DATA_PATH = "training/data/processed/grounded_conversation_v12"  # @param {type:"string"}
MODEL_ID = "Qwen/Qwen3-4B"  # @param {type:"string"}

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path(DRIVE_OUTPUT_ROOT)
else:
    OUTPUT_ROOT = Path("training/colab-runs")

if not RUN_NAME.strip() or any(part in RUN_NAME for part in ("/", "\\", "..")):
    raise ValueError("RUN_NAME phải là một tên thư mục đơn, không chứa /, \\ hoặc ..")
RUN_DIR = OUTPUT_ROOT / RUN_NAME
SOURCE_DATA = Path("training/data/reinforcement_v12.jsonl")
PROCESSED_DIR = RUN_DIR / "data"
TRAIN_DATA = PROCESSED_DIR / "grounded_conversation_train.jsonl"
VALIDATION_DATA = PROCESSED_DIR / "grounded_conversation_validation.jsonl"
TEST_DATA = PROCESSED_DIR / "grounded_conversation_test.jsonl"
PRODUCTION_R3 = Path("artifacts/travelmate-qwen3-4b-lora-v12-grounded-conversation-r3")
REGRESSION_DATA_DIR = Path(REGRESSION_DATA_PATH)
INIT_ADAPTER = Path(R2_ADAPTER_PATH) if RUN_MODE == "reproduce_v12_r3" else PRODUCTION_R3
OUTPUT_ADAPTER = RUN_DIR / "adapter"
PREDICTIONS = RUN_DIR / "predictions" / "grounded_conversation.jsonl"
REPORT = RUN_DIR / "reports" / "grounded_conversation.json"
EPOCHS = 1.0
LEARNING_RATE = "5e-6"
MAX_LENGTH = 512
MAX_NEW_TOKENS = 192

adapter_weights = INIT_ADAPTER / "adapter_model.safetensors"
if not (INIT_ADAPTER / "adapter_config.json").exists() or not adapter_weights.exists():
    raise FileNotFoundError(f"Thiếu adapter đầu vào: {INIT_ADAPTER}. Chế độ tái lập r3 yêu cầu upload adapter r2 hoặc đặt R2_ADAPTER_PATH tới Google Drive.")
if adapter_weights.stat().st_size < 1_000_000:
    raise RuntimeError(f"File trọng số có vẻ là Git LFS pointer, chưa phải adapter thật: {adapter_weights}")

print("Mode:", RUN_MODE)
print("Run directory:", RUN_DIR)
print("Train:", TRAIN_DATA)
print("Validation:", VALIDATION_DATA)
print("Test:", TEST_DATA)
print("Init adapter:", INIT_ADAPTER)
print("Candidate output:", OUTPUT_ADAPTER)

## Ô 4 — Tạo Train / Validation / Test

Tách theo trường `split` đã được builder v12 gán sẵn. Cell kiểm tra đúng 684 train, 102 validation và 102 held-out test, đồng thời ghi SHA-256 của nguồn để truy vết lần chạy. Đây là bước tái lập split từ dataset đã được builder tạo và duyệt; không thay thế công đoạn thiết kế/gán nhãn ban đầu.

In [ ]:
import hashlib
import json
from collections import Counter

records = [json.loads(line) for line in SOURCE_DATA.read_text(encoding="utf-8").splitlines() if line.strip()]
splits = {name: [record for record in records if record["split"] == name] for name in ("train", "validation", "test")}
expected = {"train": 684, "validation": 102, "test": 102}
actual = {name: len(items) for name, items in splits.items()}
if actual != expected:
    raise RuntimeError(f"Split v12 không đúng: {actual}; mong đợi {expected}")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
split_paths = {"train": TRAIN_DATA, "validation": VALIDATION_DATA, "test": TEST_DATA}
for name, items in splits.items():
    path = split_paths[name]
    path.write_text("".join(json.dumps(item, ensure_ascii=False) + "\n" for item in items), encoding="utf-8")

source_sha256 = hashlib.sha256(SOURCE_DATA.read_bytes()).hexdigest()
print("Split:", actual)
print("Source SHA-256:", source_sha256)
print("Category train:", Counter(record["category"] for record in splits["train"]))

## Ô 5 — Xem thử dữ liệu

Người train phải đọc mẫu thật trước khi dùng GPU. Thay `SAMPLE_INDEX` để xem mẫu khác.

In [ ]:
SAMPLE_INDEX = 0  # @param {type:"integer"}
sample = splits["train"][SAMPLE_INDEX]
print("ID:", sample["id"])
print("Category:", sample["category"])
print("Review:", sample["reviewStatus"])
for message in sample["messages"]:
    print(f"\n[{message['role'].upper()}]\n{message['content']}")

## Ô 6 — Validate dữ liệu

Kiểm tra JSONL, metadata, ID và trạng thái `approved`.

In [ ]:
for dataset in (TRAIN_DATA, VALIDATION_DATA, TEST_DATA):
    subprocess.run(
        [sys.executable, "-m", "training.validate_dataset", str(dataset), "--minimum-records", "1", "--require-metadata", "--require-review-status", "approved"],
        check=True,
    )
print("VALIDATION PASSED")

## Ô 7 — Dry-run

Kiểm tra tham số và số optimizer step dự kiến. Cell này chưa tải model và chưa train GPU.

In [ ]:
train_command = [
    sys.executable, "-m", "training.train_qlora",
    "--train-dataset", str(TRAIN_DATA),
    "--eval-dataset", str(VALIDATION_DATA),
    "--model-id", MODEL_ID,
    "--init-adapter-path", str(INIT_ADAPTER),
    "--output-dir", str(OUTPUT_ADAPTER),
    "--epochs", str(EPOCHS),
    "--max-length", str(MAX_LENGTH),
    "--learning-rate", LEARNING_RATE,
    "--gradient-accumulation-steps", "16",
    "--lora-r", "8", "--lora-alpha", "16",
    "--save-steps", "20",
]
subprocess.run([*train_command, "--dry-run"], check=True)

## Ô 8 — Train QLoRA thật

Bật `RUN_TRAIN`, nhập `TRAIN`, rồi mới Run cell. Cell từ chối ghi vào thư mục output đã tồn tại.

In [ ]:
RUN_TRAIN = False  # @param {type:"boolean"}
CONFIRM_TRAIN = ""  # @param {type:"string"}

if not RUN_TRAIN or CONFIRM_TRAIN != "TRAIN":
    raise RuntimeError("Train đang khóa. Bật RUN_TRAIN và nhập TRAIN để xác nhận.")
if OUTPUT_ADAPTER.exists():
    raise FileExistsError(f"Output đã tồn tại, hãy chọn tên candidate mới: {OUTPUT_ADAPTER}")
subprocess.run(train_command, check=True)
print("TRAIN COMPLETED:", OUTPUT_ADAPTER)

## Ô 9 — Sinh prediction trên held-out test

Chỉ chạy sau khi candidate đã train xong. Test không được dùng để cập nhật trọng số. Notebook từ chối dùng lại file prediction cũ để tránh trộn kết quả giữa hai adapter.

In [ ]:
RUN_GENERATION = False  # @param {type:"boolean"}
if not RUN_GENERATION:
    raise RuntimeError("Bật RUN_GENERATION sau khi đã kiểm tra đúng adapter candidate.")
if not (OUTPUT_ADAPTER / "adapter_config.json").exists():
    raise FileNotFoundError(f"Không tìm thấy adapter: {OUTPUT_ADAPTER}")
if PREDICTIONS.exists():
    raise FileExistsError(f"Prediction đã tồn tại: {PREDICTIONS}. Hãy dùng RUN_NAME mới để đánh giá candidate mới.")
PREDICTIONS.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [sys.executable, "-m", "training.generate_predictions", "--dataset", str(TEST_DATA), "--adapter-path", str(OUTPUT_ADAPTER), "--output", str(PREDICTIONS), "--max-new-tokens", str(MAX_NEW_TOKENS)],
    check=True,
)

## Ô 10 — Chấm grounded conversation

Chấm đúng tỉnh, đủ địa điểm catalog, ranh giới realtime và mức tuân thủ template.

In [ ]:
REPORT.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [sys.executable, "-m", "training.evaluate_grounded_conversation", "--dataset", str(TEST_DATA), "--predictions", str(PREDICTIONS), "--output", str(REPORT)],
    check=True,
)
report = json.loads(REPORT.read_text(encoding="utf-8"))
print(json.dumps(report, ensure_ascii=False, indent=2))

## Ô 11 — Review prediction thủ công

Evaluator không thay thế việc đọc câu trả lời. Chọn một index để kiểm tra prompt, expected answer và output thật.

In [ ]:
REVIEW_INDEX = 0  # @param {type:"integer"}
test_records = [json.loads(line) for line in TEST_DATA.read_text(encoding="utf-8").splitlines() if line.strip()]
prediction_records = [json.loads(line) for line in PREDICTIONS.read_text(encoding="utf-8").splitlines() if line.strip()]
predictions_by_id = {item["id"]: item for item in prediction_records}
case = test_records[REVIEW_INDEX]
prediction = predictions_by_id[case["id"]]
print("ID:", case["id"])
print("\nUSER:", [m["content"] for m in case["messages"] if m["role"] == "user"][-1])
print("\nEXPECTED:", case["messages"][-1]["content"])
print("\nPREDICTED:", prediction.get("prediction", prediction.get("response")))

## Ô 12 — Full adapter regression

Sinh và chấm lại nationwide structured, intent, transition, UX và reasoning bằng chính candidate mới; sau đó chạy unit test và Ruff. Các file held-out kế thừa phải có trong `REGRESSION_DATA_DIR` và không được đưa vào train. Bước này tải adapter nhiều lần nên có thể chạy lâu.

In [ ]:
RUN_FULL_REGRESSION = False  # @param {type:"boolean"}
if not RUN_FULL_REGRESSION:
    raise RuntimeError("Bật RUN_FULL_REGRESSION để đánh giá toàn bộ candidate.")

suite_specs = {
    "nationwide_structured": ("nationwide_structured_smoke_test.jsonl", "training.evaluate_nationwide", 34, 384),
    "intent": ("intent_test.jsonl", "training.evaluate_intent_execution", 11, 384),
    "transition": ("transition_test.jsonl", "training.evaluate_state_transition", 17, 384),
    "ux": ("ux_test.jsonl", "training.evaluate_natural_ux", 16, 384),
    "reasoning_raw": ("reasoning_test.jsonl", "training.evaluate_reasoning", 4, 384),
}
missing = [name for name, (filename, _, _, _) in suite_specs.items() if not (REGRESSION_DATA_DIR / filename).exists()]
if missing:
    raise FileNotFoundError(f"Thiếu held-out suite {missing} trong {REGRESSION_DATA_DIR}. Hãy upload processed bundle đã được bảo vệ khỏi train.")

regression_results = {}
for name, (filename, evaluator, minimum_passed, max_tokens) in suite_specs.items():
    dataset = REGRESSION_DATA_DIR / filename
    prediction_path = RUN_DIR / "predictions" / f"{name}.jsonl"
    report_path = RUN_DIR / "reports" / f"{name}.json"
    if prediction_path.exists() or report_path.exists():
        raise FileExistsError(f"Đã có output của suite {name}; dùng RUN_NAME mới để tránh trộn candidate.")
    prediction_path.parent.mkdir(parents=True, exist_ok=True)
    report_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, "-m", "training.generate_predictions", "--dataset", str(dataset), "--adapter-path", str(OUTPUT_ADAPTER), "--output", str(prediction_path), "--max-new-tokens", str(max_tokens)], check=True)
    subprocess.run([sys.executable, "-m", evaluator, "--dataset", str(dataset), "--predictions", str(prediction_path), "--output", str(report_path)], check=True)
    suite_report = json.loads(report_path.read_text(encoding="utf-8"))
    passed = int(suite_report.get("passed", 0))
    regression_results[name] = {"passed": passed, "minimum": minimum_passed, "gatePassed": passed >= minimum_passed}

subprocess.run([sys.executable, "-m", "pytest"], check=True)
subprocess.run([sys.executable, "-m", "ruff", "check", "."], check=True)
regression_gate_passed = all(item["gatePassed"] for item in regression_results.values())
regression_summary_path = RUN_DIR / "reports" / "regression_summary.json"
regression_summary_path.write_text(json.dumps(regression_results, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(regression_results, ensure_ascii=False, indent=2))
if not regression_gate_passed:
    raise RuntimeError("Candidate làm giảm ít nhất một held-out suite; không được promote.")
print("FULL REGRESSION PASSED")

## Ô 13 — Cổng promote và kết luận

Không tự động sửa `.env`. `PROMOTE` chỉ hợp lệ khi grounding đạt tuyệt đối ở ba metric nội dung, full regression đạt, prediction đã được review thủ công và demo runtime đạt 20/20.

In [ ]:
MANUAL_REVIEW_COMPLETE = False  # @param {type:"boolean"}
DEMO_RUNTIME_RESULT = "NOT_RUN"  # @param ["NOT_RUN", "PASS_20_20", "FAILED"]
DECISION = "REJECT"  # @param ["REJECT", "PROMOTE"]
NOTES = "Chưa review thủ công toàn bộ prediction."  # @param {type:"string"}
grounded_report = json.loads(REPORT.read_text(encoding="utf-8"))
grounded_metrics = grounded_report["metrics"]
grounding_gate_passed = all(
    grounded_metrics[name]["passed"] == grounded_metrics[name]["total"] == 102
    for name in ("currentProvince", "allCatalogPlaces", "realtimeBoundary")
)
regression_summary_path = RUN_DIR / "reports" / "regression_summary.json"
if not regression_summary_path.exists():
    raise FileNotFoundError("Chưa có regression_summary.json; hãy chạy Ô 12.")
regression_results = json.loads(regression_summary_path.read_text(encoding="utf-8"))
regression_gate_passed = all(item["gatePassed"] for item in regression_results.values())
promotion_eligible = all((grounding_gate_passed, regression_gate_passed, MANUAL_REVIEW_COMPLETE, DEMO_RUNTIME_RESULT == "PASS_20_20"))
if DECISION == "PROMOTE" and not promotion_eligible:
    raise RuntimeError("Không thể PROMOTE: grounding, regression, manual review hoặc demo runtime chưa đạt.")
summary = {
    "runMode": RUN_MODE,
    "runName": RUN_NAME,
    "modelId": MODEL_ID,
    "initAdapter": str(INIT_ADAPTER),
    "candidateAdapter": str(OUTPUT_ADAPTER),
    "trainDataset": str(TRAIN_DATA),
    "validationDataset": str(VALIDATION_DATA),
    "testDataset": str(TEST_DATA),
    "report": str(REPORT),
    "sourceDatasetSha256": source_sha256,
    "groundingGatePassed": grounding_gate_passed,
    "regressionGatePassed": regression_gate_passed,
    "manualReviewComplete": MANUAL_REVIEW_COMPLETE,
    "demoRuntimeResult": DEMO_RUNTIME_RESULT,
    "promotionEligible": promotion_eligible,
    "decision": DECISION,
    "notes": NOTES,
}
summary_path = RUN_DIR / "candidate_summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))